# 🍽️ FoodTrend — TikTok Food Content Strategy Dashboard

> **One-hand friendly**: Just press **Shift+Enter** through each cell. No typing needed!
>
> *For aspiring food content creators who want data-driven strategy.*

In [ ]:
# Setup (Shift+Enter to run)
import sys; sys.path.insert(0, 'src')
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from data_engine import *

SEED = 42  # Change for different data
heatmap = generate_posting_heatmap(seed=SEED)
best_times = find_best_posting_times(heatmap)
competitors = generate_competitor_data(seed=SEED)
growth = generate_growth_projection(500, 12, 4, seed=SEED)
gaps = find_content_gaps(competitors)
hashtags = generate_hashtag_data(seed=SEED)
print('✅ Data loaded! Keep pressing Shift+Enter ➡️')

## 📅 When to Post — Engagement Heatmap

In [ ]:
fig = go.Figure(data=go.Heatmap(
    z=heatmap, x=[f'{h}:00' for h in range(24)], y=DAYS,
    colorscale='YlOrRd', hovertemplate='%{y} at %{x}: %{z:.0%}<extra></extra>',
))
fig.update_layout(title='📅 Best Times to Post Food Content',
    xaxis_title='Hour', height=350, template='plotly_dark',
    paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e', font=dict(family='Georgia'))
fig.show()

print('\n⏰ Top 5 Posting Slots:')
for t in best_times:
    print(f"  {t['day']:>9} at {t['hour']:2d}:00 — {t['score']:.0%} engagement")

## 🔥 Trending Food Categories — Growth Rate

In [ ]:
cat_df = pd.DataFrame(FOOD_CATEGORIES).sort_values('growth', ascending=True)
colors = ['#ff6b6b' if g > 25 else '#feca57' if g > 15 else '#48dbfb' for g in cat_df['growth']]

fig = go.Figure(go.Bar(x=cat_df['growth'], y=[f"{r['icon']} {r['name']}" for _, r in cat_df.iterrows()],
    orientation='h', marker_color=colors,
    text=[f"+{g}%" for g in cat_df['growth']], textposition='outside'))
fig.update_layout(title='🔥 Food Category Growth Rate (2026)',
    xaxis_title='Growth %', height=400, template='plotly_dark',
    paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e', font=dict(family='Georgia'))
fig.show()

## 🏷️ Trending Hashtags

In [ ]:
ht_df = pd.DataFrame(hashtags[:12])
comp_colors = {'high': '#ff6b6b', 'medium': '#feca57', 'low': '#48dbfb'}

fig = go.Figure()
fig.add_trace(go.Bar(x=ht_df['hashtag'], y=ht_df['growth_7d'], name='7-Day Growth %',
    marker_color=[comp_colors[c] for c in ht_df['competition']],
    text=[f"+{g}%" for g in ht_df['growth_7d']], textposition='outside'))
fig.update_layout(title='🏷️ Top Hashtags by Growth (Red=High Competition, Blue=Low)',
    yaxis_title='Growth %', height=400, template='plotly_dark',
    paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e', font=dict(family='Georgia'),
    xaxis_tickangle=-45)
fig.show()

## 👥 Competitor Benchmarks

In [ ]:
comp_df = pd.DataFrame(competitors)

fig = px.scatter(comp_df, x='followers', y='engagement_rate', size='avg_views',
    color='growth_30d', text='name', hover_data=['niche', 'posts_per_week'],
    color_continuous_scale='YlOrRd', size_max=40,
    labels={'followers': 'Followers', 'engagement_rate': 'Engagement Rate %', 'growth_30d': '30d Growth %'})
fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(title='👥 Competitor Landscape (Size = Avg Views, Color = Growth)',
    height=450, template='plotly_dark',
    paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e', font=dict(family='Georgia'))
fig.show()

## 🎯 Content Gap Analysis — Your Opportunity

In [ ]:
print('🎯 Underserved Categories (Your Opportunity):\n')
for g in gaps:
    opp_icon = '🟢' if g['opportunity'] == 'high' else '🟡'
    print(f"  {opp_icon} {g['icon']} {g['name']}")
    print(f"     Growth: +{g['growth']}% | Opportunity: {g['opportunity']}")
    print()

## 📈 12-Month Growth Projection

In [ ]:
gr_df = pd.DataFrame(growth)

fig = go.Figure()
fig.add_trace(go.Scatter(x=gr_df['month'], y=gr_df['followers'], mode='lines+markers',
    name='Followers', line=dict(color='#ff6b6b', width=3), marker=dict(size=8)))

# Highlight viral months
viral = gr_df[gr_df['viral_month']]
if len(viral) > 0:
    fig.add_trace(go.Scatter(x=viral['month'], y=viral['followers'], mode='markers',
        name='Viral Month! 🚀', marker=dict(size=16, color='#feca57', symbol='star')))

fig.update_layout(title=f"📈 Projected Growth (Start: 500 → End: {growth[-1]['followers']:,})",
    xaxis_title='Month', yaxis_title='Followers', height=400, template='plotly_dark',
    paper_bgcolor='#1a1a2e', plot_bgcolor='#16213e', font=dict(family='Georgia'))
fig.show()
print(f"\nProjected followers after 12 months: {growth[-1]['followers']:,}")
print(f"Viral months: {sum(1 for g in growth if g['viral_month'])}")

## 📋 Your Strategy Summary

In [ ]:
print(get_strategy_summary(competitors, gaps, best_times))

---
*FoodTrend — Data-driven content strategy for food creators. Change `SEED` at the top for different data.*

*One hand, one Shift+Enter at a time.* 🍳